# Airtable Passthrough and Derivative Audit

Read-only audit of the `cfhe-data` repository and the live CFHE Airtable base.

## tl;dr

The repaired Current, housing-element, Builder’s Remedy, Rent, RHNA Prediction, Reports, County, Census, and Event-email derivative chains reconcile independently. The only classified source-normalization issue is 47 Events that retain case-only email repeats; exact sets and exact-token uniqueness still pass.

## Context & Methods

The audit reads Airtable schema, records, interfaces, forms, views, and automation configuration without calling mutation endpoints. It independently recomputes links, counts, rollups, Current-gated housing element passthroughs, zero-safe formulas, Census keys, exact Event email sets and duplicate tokens, nine County interface pages, repository derivatives, and the production sync dry run. The notebook loads the aggregate result emitted by `airtable_derivatives_audit.py`; set `REFRESH_LIVE` below to rerun that collection step.

In [1]:
import json
import subprocess
from pathlib import Path

cursor = Path.cwd().resolve()
for candidate in (cursor, *cursor.parents):
    if (candidate / "pyproject.toml").exists() and (
        candidate / "src" / "cfhe_data"
    ).exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the cfhe-data repository root")

REFRESH_LIVE = False
if REFRESH_LIVE:
    subprocess.run(
        [
            "uv",
            "run",
            "--with",
            "requests",
            "--with-editable",
            ".",
            "--extra",
            "dev",
            "python",
            "docs/audits/airtable_derivatives_audit.py",
        ],
        cwd=REPO_ROOT,
        check=True,
    )

result_path = REPO_ROOT / "data" / "airtable" / "derivatives_audit_results.json"
audit = json.loads(result_path.read_text(encoding="utf-8"))
print(f"Loaded read-only audit executed at {audit['executed_at']}")

Loaded read-only audit executed at 2026-08-27T17:45:12.940313+00:00


## Data

In [2]:
summary = audit["summary"]
record_total = sum(summary["table_record_counts"].values())
coverage = {
    "Airtable tables": summary["table_count"],
    "Airtable records": record_total,
    "Fields": summary["field_count"],
    "Computed fields": summary["computed_field_count"],
    "Linked-record fields": summary["link_field_count"],
    "Views": summary["view_count"],
    "Interface pages": summary["interface_page_count"],
    "Automations": summary["automation_count"],
}
for label, value in coverage.items():
    print(f"{label:24} {value:,}")
print(f"Managed permit total      {summary['live_permit_total']:,}")
print(f"Reviewed source total     {summary['desired_permit_total']:,}")

Airtable tables          14
Airtable records         8,475
Fields                   267
Computed fields          114
Linked-record fields     43
Views                    49
Interface pages          12
Automations              14
Managed permit total      501,601
Reviewed source total     501,601


## Results

In [3]:
print("Overall status:", summary["overall_status"].upper())
print("Checks by status:", summary["check_counts"])
print("Issues by severity:", summary["issue_severity_counts"])

rank = {"high": 0, "medium": 1, "low": 2, "info": 3}
nonpass = sorted(
    (check for check in audit["checks"] if check["status"] != "pass"),
    key=lambda check: (rank[check["severity"]], check["domain"], check["check"]),
)
print("\n| Severity | Domain | Check | Instances | Evidence |")
print("|---|---|---|---:|---|")
for check in nonpass:
    evidence = check["evidence"].replace("|", "/")
    print(
        f"| {check['severity']} | {check['domain']} | {check['check']} | {check['failures']:,} | {evidence} |"
    )

Overall status: WARN
Checks by status: {'pass': 46, 'warn': 2}
Issues by severity: {'low': 2}

| Severity | Domain | Check | Instances | Evidence |
|---|---|---|---:|---|
| low | automations | automation deployment inventory | 1 | 13 deployed and 1 undeployed automations |
| low | event email chain | case-normalized email repeats are classified as source data | 47 | 47 Events retain case-only repeats from the Watchdogs source; exact sets and exact-token uniqueness remain valid |


In [4]:
checks_by_name = {check["check"]: check for check in audit["checks"]}
core_assertions = [
    (
        "Live permit total equals reviewed source",
        summary["live_permit_total"] == summary["desired_permit_total"],
    ),
    (
        "Production dry run reports zero changes",
        summary["sync_summary"].get("change_count") == 0,
    ),
    (
        "All intended jurisdictions matched",
        summary["sync_summary"].get("matched_count") == 539,
    ),
    (
        "Current cycle split is 522 sixth and 17 seventh",
        summary["current_cycle_counts"] == {"6th": 522, "7th": 17},
    ),
    (
        "Current helpers and HE passthroughs have zero mismatches",
        summary["current_helper_mismatch_count"] == 0
        and summary["he_passthrough_mismatch_count"] == 0,
    ),
    (
        "Both Builder outputs have zero mismatches",
        summary["builder_text_mismatch_count"] == 0
        and summary["builder_flag_mismatch_count"] == 0,
    ),
    (
        "Rent outputs have zero mismatches",
        summary["rent_overview_mismatch_count"] == 0
        and summary["rent_text_mismatch_count"] == 0,
    ),
    (
        "Prediction, lookup, and color have zero mismatches",
        summary["rhna_prediction_mismatch_count"] == 0
        and summary["prediction_lookup_mismatch_count"] == 0
        and summary["prediction_color_mismatch_count"] == 0,
    ),
    (
        "Reports ratios have zero mismatches and errors",
        summary["pro_housing_mismatch_count"] == 0
        and summary["pro_housing_error_count"] == 0,
    ),
    (
        "Census keys are unique",
        summary["census_link_violation_count"] == 0
        and summary["census_duplicate_group_count"] == 0,
    ),
    (
        "Event exact sets and exact-token uniqueness pass",
        summary["event_lookup_set_mismatch_count"] == 0
        and summary["event_unique_formula_set_mismatch_count"] == 0
        and summary["event_unique_exact_duplicate_token_event_count"] == 0,
    ),
    (
        "Counties interface has nine successful page probes",
        summary["county_interface_page_count"] == 9
        and summary["county_interface_probe_success_count"] == 9,
    ),
]
for label, passed in core_assertions:
    print(("PASS" if passed else "FAIL").ljust(6), label)

PASS   Live permit total equals reviewed source
PASS   Production dry run reports zero changes
PASS   All intended jurisdictions matched
PASS   Current cycle split is 522 sixth and 17 seventh
PASS   Current helpers and HE passthroughs have zero mismatches
PASS   Both Builder outputs have zero mismatches
PASS   Rent outputs have zero mismatches
PASS   Prediction, lookup, and color have zero mismatches
PASS   Reports ratios have zero mismatches and errors
PASS   Census keys are unique
PASS   Event exact sets and exact-token uniqueness pass
PASS   Counties interface has nine successful page probes


In [5]:
repaired_evidence = {
    "Current RHNA rows": summary["current_rhna_count"],
    "Current helper mismatches": summary["current_helper_mismatch_count"],
    "HE passthrough mismatches": summary["he_passthrough_mismatch_count"],
    "Builder text + flag mismatches": summary["builder_text_mismatch_count"]
    + summary["builder_flag_mismatch_count"],
    "Rent mismatches": summary["rent_overview_mismatch_count"]
    + summary["rent_text_mismatch_count"],
    "Prediction-chain mismatches": summary["rhna_prediction_mismatch_count"]
    + summary["prediction_lookup_mismatch_count"]
    + summary["prediction_color_mismatch_count"],
    "Reports mismatches + errors": summary["pro_housing_mismatch_count"]
    + summary["pro_housing_error_count"],
    "Census duplicate groups": summary["census_duplicate_group_count"],
    "Event exact-set mismatches": summary["event_lookup_set_mismatch_count"]
    + summary["event_unique_formula_set_mismatch_count"],
    "Event exact duplicate-token records": summary[
        "event_unique_exact_duplicate_token_event_count"
    ],
    "Event case-only source repeats": summary[
        "event_unique_case_normalized_duplicate_token_event_count"
    ],
    "Successful County interface probes": summary[
        "county_interface_probe_success_count"
    ],
}
for label, value in repaired_evidence.items():
    print(f"{label:42} {value:,}")

Current RHNA rows                          539
Current helper mismatches                  0
HE passthrough mismatches                  0
Builder text + flag mismatches             0
Rent mismatches                            0
Prediction-chain mismatches                0
Reports mismatches + errors                0
Census duplicate groups                    0
Event exact-set mismatches                 0
Event exact duplicate-token records        0
Event case-only source repeats             47
Successful County interface probes         9


## Takeaways

1. The Current flag is cycle agnostic and resolves 522 sixth-cycle and 17 seventh-cycle jurisdictions through the same helper and rollup chain.
2. Housing element, Builder’s Remedy, Rent, Prediction, Reports, County, Census, and Event derivatives are now checked by independent value recomputation, not only by Airtable schema validity.
3. Event email validation uses exact source sets and exact duplicate-token counts, so it detects truncation without reinstating an arbitrary threshold.
4. Case-only email repeats remain classified as source normalization because the derivative chain preserves the exact source spellings.
5. Interface layout and automation configuration are inspectable, but automation run history and connected-account health remain outside this audit identity’s access.